In [0]:
# ============================================
# GOLD LAYER - DIM DATE
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.types import DateType
from datetime import datetime, timedelta

df_silver = spark.table("silver.transactions_clean")

# Extract min/max date from transactions
date_bounds = df_silver.select(
    min(to_date("date")).alias("min_date"),
    max(to_date("date")).alias("max_date")
).collect()[0]

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

print("Date range:", min_date, "to", max_date)


Date range: 2010-01-01 to 2019-10-31


In [0]:
# Generate date sequence
date_list = [
    (min_date + timedelta(days=i),)
    for i in range((max_date - min_date).days + 1)
]

df_dates = spark.createDataFrame(date_list, ["date"])


In [0]:
# Add date attributes
dim_date = (
    df_dates
    .withColumn("date_sk", date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("week_of_year", weekofyear("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("is_weekend",
        when(dayofweek("date").isin([1,7]), True).otherwise(False)
    )
)


In [0]:
# Write Gold table
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

dim_date.write.mode("overwrite").format("delta").saveAsTable(
    "gold.dim_date"
)
